# Kitsune-TTS — CPU / T4 benchmark

Reproducible FP32 benchmark for the published V1 artifacts. It records Colab hardware, model load time, first complete synthesis after load, warmed model-only latency, end-to-end latency and RTF.

It tests the FP32 and FP16 *files* (both execute in FP32 in the Python API), the optional PyTorch `fast_cpu` vocoder on CPU, ONNX Runtime CPU, and PyTorch CUDA. `noise_scale=0` makes the output deterministic within each backend.

> `first_synthesis_s` means the time to receive the complete waveform after the model is loaded. Kitsune does not stream samples yet, so this is not literal time-to-first-sample. Do not compare first-run times across backends as a pure model-speed number: runtime initialization and kernel caching matter.

In [ ]:
# Runtime setup. Run once, then Runtime > Restart session if Colab asks for it.
%pip -q install --upgrade huggingface_hub pandas tabulate psutil onnxruntime
!apt-get -qq update && apt-get -qq install -y espeak-ng


In [ ]:
# Hardware and runtime report — copy this output into the eventual results table.
import json, os, platform, subprocess, sys
import psutil
import torch

def command_output(command):
    try:
        return subprocess.check_output(command, shell=True, text=True, stderr=subprocess.STDOUT).strip()
    except subprocess.CalledProcessError as error:
        return error.output.strip()

hardware = {
    'platform': platform.platform(),
    'python': sys.version,
    'cpu': command_output('lscpu') or platform.processor(),
    'ram': command_output('free -h'),
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'torch_cuda': torch.version.cuda,
    'gpu': command_output('nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader'),
    'process_rss_at_start_mb': round(psutil.Process().memory_info().rss / 2**20, 2),
}
print(json.dumps(hardware, indent=2))


## Upload the source ZIP generated locally

Upload `kitsune_tts_fast_cpu_source.zip` from `experiments/fast-cpu-tests/`. The ZIP intentionally excludes `model/`, virtual environments, Git metadata, cached experiment output and large artifacts.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

uploaded = files.upload()
zip_name = next((name for name in uploaded if name.endswith('.zip')), None)
if zip_name is None:
    raise RuntimeError('Upload the Kitsune source ZIP, not a checkpoint or WAV.')
WORKDIR = Path('/content/Kitsune-TTS')
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
WORKDIR.mkdir()
shutil.unpack_archive(zip_name, WORKDIR)
print('Source extracted to', WORKDIR)
assert (WORKDIR / 'kitsune' / 'api.py').is_file()


In [ ]:
# Download only the published artifacts required for the matrix.
from huggingface_hub import hf_hub_download

REPO_ID = 'Heitorkk2/Kitsune-TTS-V1'
MODEL_DIR = WORKDIR / 'model'
MODEL_DIR.mkdir(exist_ok=True)
for filename in ('model_config.json', 'latest_model_fp16.pth', 'latest_model_fp32.pth', 'kitsune39M.onnx'):
    try:
        path = hf_hub_download(REPO_ID, filename, local_dir=MODEL_DIR)
        print('Downloaded:', path)
    except Exception as error:
        print(f'SKIP {filename}: {error}')

required = ['model_config.json', 'latest_model_fp16.pth', 'latest_model_fp32.pth']
missing = [name for name in required if not (MODEL_DIR / name).is_file()]
if missing:
    raise RuntimeError(f'Missing required model files: {missing}')


In [ ]:
# Benchmark configuration. Keep noise_scale at zero for reproducible comparisons.
import gc, importlib, importlib.metadata, time
import numpy as np
import pandas as pd
import torch
import sys
sys.path.insert(0, str(WORKDIR))
from kitsune.api import KitsuneSynthesizer

TEXT = 'Olá! Hoje eu vou contar uma pequena história. A noite estava tranquila, mas uma luz apareceu entre as árvores. Você também conseguiu enxergar?'
SPEAKER = 'frieren'
LANG = 'pt-br'
NOISE_SCALE = 0.0
LENGTH_SCALE = 1.0
WARM_RUNS = 1
MEASURE_RUNS = 5
CPU_THREADS = min(8, os.cpu_count() or 1)
torch.set_num_threads(CPU_THREADS)
print({'cpu_threads': CPU_THREADS, 'torch': torch.__version__, 'onnxruntime': importlib.metadata.version('onnxruntime')})


In [ ]:
# Helpers. model_only time excludes phonemization and NumPy/WAV writes.
PROCESS = psutil.Process()

def rss_mb():
    return PROCESS.memory_info().rss / 2**20

def directory_size_mb(path):
    return sum(file.stat().st_size for file in Path(path).rglob('*') if file.is_file()) / 2**20

def imported_module_size_mb(name):
    module = importlib.import_module(name)
    path = Path(module.__file__).resolve()
    return directory_size_mb(path.parent) if path.name == '__init__.py' else path.stat().st_size / 2**20

def deployment_disk_mb(backend, artifact):
    modules = ['numpy', 'phonemizer', 'num2words', 'torch' if backend == 'torch' else 'onnxruntime']
    sizes = {'artifact_mb': Path(artifact).stat().st_size / 2**20, 'kitsune_source_mb': directory_size_mb(WORKDIR / 'kitsune')}
    for name in modules:
        sizes[f'package_{name}_mb'] = imported_module_size_mb(name)
    sizes['approx_model_plus_runtime_mb'] = sum(value for key, value in sizes.items() if key.endswith('_mb'))
    return sizes

def gpu_process_mb():
    if not torch.cuda.is_available(): return None
    try:
        lines = subprocess.check_output(['nvidia-smi', '--query-compute-apps=pid,used_memory', '--format=csv,noheader,nounits'], text=True).splitlines()
        for line in lines:
            pid, memory = [part.strip() for part in line.split(',', 1)]
            if int(pid) == os.getpid(): return float(memory)
    except Exception: pass
    return None
def synchronize(device):
    if device == 'cuda':
        torch.cuda.synchronize()

def model_only(synth, text, speaker, lang=LANG):
    sequence = synth._text_to_sequence(text, lang)
    sid_value = synth._resolve_speaker(speaker)
    if synth.backend == 'torch':
        x = torch.tensor(sequence, dtype=torch.long, device=synth.device).unsqueeze(0)
        lengths = torch.tensor([len(sequence)], dtype=torch.long, device=synth.device)
        sid = torch.tensor([sid_value], dtype=torch.long, device=synth.device)
        with torch.inference_mode():
            audio = synth.model.infer(x, lengths, sid=sid, noise_scale=NOISE_SCALE, length_scale=LENGTH_SCALE)[0]
        synchronize(synth.device.type)
        return audio[0, 0].detach().cpu().numpy()
    x = np.asarray(sequence, dtype=np.int64)[None, :]
    feed = {'x': x, 'x_lengths': np.asarray([x.shape[1]], dtype=np.int64)}
    inputs = synth._ort_input_names
    if 'sid' in inputs: feed['sid'] = np.asarray([sid_value], dtype=np.int64)
    if 'noise_scale' in inputs: feed['noise_scale'] = np.asarray([NOISE_SCALE], dtype=np.float32)
    if 'length_scale' in inputs: feed['length_scale'] = np.asarray([LENGTH_SCALE], dtype=np.float32)
    return synth.ort.run([synth._ort_output_name], feed)[0][0, 0].astype(np.float32, copy=False)

def timed(call, device='cpu'):
    synchronize(device)
    started = time.perf_counter()
    result = call()
    synchronize(device)
    return result, time.perf_counter() - started

def run_case(name, *, checkpoint=None, onnx_path=None, device=None, fast_cpu=False, providers=None):
    artifact = checkpoint or onnx_path
    backend = 'torch' if checkpoint else 'onnx'
    row = {'case': name, 'checkpoint_file': Path(checkpoint).name if checkpoint else None,
           'onnx_file': Path(onnx_path).name if onnx_path else None, 'requested_device': device,
           'fast_cpu': fast_cpu, 'status': 'ok'}
    row.update(deployment_disk_mb(backend, artifact))
    row['rss_before_mb'] = rss_mb()
    if device == 'cuda':
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    synth = None
    try:
        synth, row['load_s'] = timed(lambda: KitsuneSynthesizer(
            checkpoint=checkpoint, onnx_path=onnx_path, device=device, fast_cpu=fast_cpu,
            ort_threads=CPU_THREADS if onnx_path and providers == ['CPUExecutionProvider'] else 0,
            providers=providers), 'cuda' if device == 'cuda' else 'cpu')
        row['rss_after_load_mb'] = rss_mb()
        row['ram_delta_load_mb'] = row['rss_after_load_mb'] - row['rss_before_mb']
        row['gpu_peak_torch_allocated_mb_after_load'] = torch.cuda.max_memory_allocated() / 2**20 if device == 'cuda' else None
        row['gpu_process_mb_after_load'] = gpu_process_mb()
        run_device = synth.device.type if synth.backend == 'torch' else 'cpu'
        torch.manual_seed(2026)
        first_audio, row['first_synthesis_s'] = timed(
            lambda: synth.synthesize(TEXT, speaker=SPEAKER, lang=LANG, noise_scale=NOISE_SCALE, length_scale=LENGTH_SCALE), run_device)
        row['rss_after_first_mb'] = rss_mb()
        row['ram_delta_first_mb'] = row['rss_after_first_mb'] - row['rss_before_mb']
        row['gpu_peak_torch_allocated_mb_after_first'] = torch.cuda.max_memory_allocated() / 2**20 if device == 'cuda' else None
        row['gpu_process_mb_after_first'] = gpu_process_mb()
        for _ in range(WARM_RUNS):
            torch.manual_seed(2026); model_only(synth, TEXT, SPEAKER)
        model_times, e2e_times = [], []
        for _ in range(MEASURE_RUNS):
            torch.manual_seed(2026); audio, elapsed = timed(lambda: model_only(synth, TEXT, SPEAKER), run_device); model_times.append(elapsed)
            torch.manual_seed(2026); _, elapsed = timed(lambda: synth.synthesize(TEXT, speaker=SPEAKER, lang=LANG, noise_scale=NOISE_SCALE, length_scale=LENGTH_SCALE), run_device); e2e_times.append(elapsed)
        if not np.isfinite(first_audio).all() or not np.isfinite(audio).all():
            raise RuntimeError('non-finite waveform')
        row.update({'backend': synth.backend, 'actual_device': run_device, 'providers': str(synth.ort.get_providers()) if synth.backend == 'onnx' else None,
                    'audio_s': len(audio) / synth.sample_rate, 'model_median_s': float(np.median(model_times)),
                    'e2e_median_s': float(np.median(e2e_times)), 'model_runs_s': model_times, 'e2e_runs_s': e2e_times,
                    'rss_after_benchmark_mb': rss_mb(), 'gpu_process_mb_after_benchmark': gpu_process_mb()})
        row['rtf'] = row['model_median_s'] / row['audio_s']
        row['first_rtf'] = row['first_synthesis_s'] / row['audio_s']
        return row, first_audio
    except Exception as error:
        row.update({'status': 'skipped_or_failed', 'error': f'{type(error).__name__}: {error}'})
        return row, None
    finally:
        del synth
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
# Full matrix: ONNX Runtime is measured on CPU; CUDA rows use PyTorch.
fp32 = str(MODEL_DIR / 'latest_model_fp32.pth')
fp16 = str(MODEL_DIR / 'latest_model_fp16.pth')
onnx = str(MODEL_DIR / 'kitsune39M.onnx')
cases = [
    ('pytorch_fp32_cpu', dict(checkpoint=fp32, device='cpu')),
    ('pytorch_fp16_file_cpu', dict(checkpoint=fp16, device='cpu')),
    ('pytorch_fp32_fast_cpu', dict(checkpoint=fp32, device='cpu', fast_cpu=True)),
    ('pytorch_fp16_file_fast_cpu', dict(checkpoint=fp16, device='cpu', fast_cpu=True)),
]
if Path(onnx).is_file():
    cases.append(('onnx_cpu', dict(onnx_path=onnx, providers=['CPUExecutionProvider'])))
if torch.cuda.is_available():
    cases.extend([
        ('pytorch_fp32_cuda', dict(checkpoint=fp32, device='cuda')),
        ('pytorch_fp16_file_cuda', dict(checkpoint=fp16, device='cuda')),
    ])

rows, audio_examples = [], {}
for name, kwargs in cases:
    print('Running', name, flush=True)
    row, audio = run_case(name, **kwargs)
    rows.append(row)
    if audio is not None:
        audio_examples[name] = audio

results = pd.DataFrame(rows)
display_columns = ['case', 'status', 'backend', 'actual_device', 'fast_cpu', 'artifact_mb', 'approx_model_plus_runtime_mb', 'rss_before_mb', 'ram_delta_load_mb', 'ram_delta_first_mb', 'gpu_process_mb_after_first', 'load_s', 'first_synthesis_s', 'audio_s', 'model_median_s', 'e2e_median_s', 'rtf', 'first_rtf', 'error']
display(results.reindex(columns=display_columns).sort_values('case'))


In [ ]:
# Save machine-readable results, a shareable HTML table, and selected audio outputs.
from scipy.io import wavfile
from IPython.display import Audio, display

OUTPUT = WORKDIR / 'benchmark_output'
OUTPUT.mkdir(exist_ok=True)
results.to_csv(OUTPUT / 'results.csv', index=False)
results.to_json(OUTPUT / 'results.json', orient='records', indent=2)
(OUTPUT / 'hardware.json').write_text(json.dumps(hardware, indent=2), encoding='utf-8')
table = results.reindex(columns=display_columns).sort_values('case').to_html(index=False, float_format=lambda value: f'{value:.5f}')
(OUTPUT / 'results.html').write_text('<!doctype html><meta charset="utf-8"><title>Kitsune benchmark</title><style>body{font-family:system-ui;margin:2rem}table{border-collapse:collapse}td,th{border:1px solid #bbb;padding:.45rem;text-align:left}</style><h1>Kitsune-TTS benchmark</h1>' + table, encoding='utf-8')
for name, audio in audio_examples.items():
    wavfile.write(OUTPUT / f'{name}.wav', 22050, audio.astype(np.float32))

# Hear CPU original versus fast_cpu if both cases completed.
for name in ('pytorch_fp32_cpu', 'pytorch_fp32_fast_cpu'):
    if name in audio_examples:
        print(name)
        display(Audio(audio_examples[name], rate=22050))

shutil.make_archive('/content/kitsune_benchmark_results', 'zip', OUTPUT)
print('Saved:', OUTPUT)


In [ ]:
# Download this ZIP after reviewing the table. It contains CSV, JSON, HTML, hardware info and WAV examples.
from google.colab import files
files.download('/content/kitsune_benchmark_results.zip')


## How to turn this into a publishable table

Use `results.csv` for the raw figures. Report the exact CPU/GPU, Colab runtime, PyTorch and ONNX Runtime versions, threads, text, speaker, model-file name, median warmed latency, RTF and whether the result is CPU or CUDA. Keep first-synthesis time in a separate column. RAM deltas are current-process RSS deltas, not total system RAM. GPU process memory and `gpu_peak_torch_allocated` are reported for PyTorch CUDA rows.

The approximate disk footprint is the model artifact + Kitsune source + imported package directories. It deliberately excludes Python itself, OS libraries, `espeak-ng`, caches and unrelated shared dependencies. For the Ryzen 5700U run, use the local Python script with the same source ZIP, weights, text, `noise_scale`, speaker and number of repetitions. Run while idle and repeat the full benchmark at least twice; CPU frequency and background load can noticeably change results. `fast_cpu` is PyTorch CPU-only and must not be presented as an ONNX or JavaScript speedup.